In [ ]:
import os
import zipfile
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

ZIP_PATH_UNITY = '/content/drive/MyDrive/FYP_ML/unity_train.zip' 
ZIP_PATH_REAL  = '/content/drive/MyDrive/FYP_ML/argaze_real.zip' 
CSV_PATH_UNITY = '/content/unity_fixed.csv'
CSV_PATH_REAL  = '/content/drive/MyDrive/FYP_ML/labels_real_full.csv'
EXTRACT_UNITY = '/content/dataset_unity'
EXTRACT_REAL = '/content/dataset_real'

# 2.data unzipping for preparation
def unzip_data(zip_path, dest_path):
    if not os.path.exists(zip_path):
        print(f"ERROR: Could not find {zip_path}.")
        return
    
    # re-run zip extract save
    if os.path.exists(dest_path):
        print(f"{os.path.basename(zip_path)} already extracted.")
        return
    print(f"Unzipping {os.path.basename(zip_path)}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dest_path)
    print(f"Extracted to {dest_path}")

unzip_data(ZIP_PATH_UNITY, EXTRACT_UNITY)
unzip_data(ZIP_PATH_REAL, EXTRACT_REAL)

Mounted at /content/drive
Unzipping unity_train.zip...
Extracted to /content/dataset_unity
Unzipping argaze_real.zip...
Extracted to /content/dataset_real


In [3]:
import os
import pandas as pd

print("--- 1. CSV CHECK ---")
csv_path = '/content/drive/MyDrive/FYP_ML/labels_real_full.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"CSV Row 0 says path is: {df.iloc[0, 0]}")
else:
    print("CSV not found.")

# 2. Inspect the "Real" Folder Structure
print("\n--- 2. FILE SYSTEM CHECK ---")
real_root = '/content/dataset_real'

found_one = False
# Walk through the folder to find the ACTUAL path of the first image
for root, dirs, files in os.walk(real_root):
    for file in files:
        if file.endswith(".png") or file.endswith(".jpg"):
            full_path = os.path.join(root, file)
            print(f"FOUND ACTUAL IMAGE AT: {full_path}")
            
            # Show us the difference
            print(f"Root dir is: {root}")
            found_one = True
            break
    if found_one:
        break

if not found_one:
    print("NO IMAGES FOUND in /content/dataset_real. Did the unzip work?")

--- 1. CSV CHECK ---
CSV Row 0 says path is: D:/projects/academical/realdata_FYP/osfstorage-archive\P1\P1_S1\P1_S1_C1\0.png

--- 2. FILE SYSTEM CHECK ---
FOUND ACTUAL IMAGE AT: /content/dataset_real/P18/P18_S2/P18_S2_C1/14065.png
Root dir is: /content/dataset_real/P18/P18_S2/P18_S2_C1


In [ ]:
import os
import pandas as pd

print("--- UNITY DATA DIAGNOSTIC ---")

# 1. Check the CSV
unity_csv_path = '/content/drive/MyDrive/FYP_ML/unity_fixed.csv'
if os.path.exists(unity_csv_path):
    df = pd.read_csv(unity_csv_path)
    print(f"CSV Row 0 says filename is: {df.iloc[0, 0]}")
else:
    print("Unity CSV not found.")

# 2. Hunt for the actual files
search_root = '/content/dataset_unity'
found = False

print(f"\nScanning {search_root} for images...")
for root, dirs, files in os.walk(search_root):
    for file in files:
        if file.endswith(".jpg") or file.endswith(".png"):
            # Found one! Print its full path
            print(f"FOUND ACTUAL IMAGE AT: {os.path.join(root, file)}")
            found = True
            break # Stop after finding one
    if found:
        break

if not found:
    print("NO IMAGES FOUND. The unzip might have failed or the folder is empty.")

--- UNITY DATA DIAGNOSTIC ---
CSV Row 0 says filename is: 1.jpg

Scanning /content/dataset_unity for images...
✅ FOUND ACTUAL IMAGE AT: /content/dataset_unity/3984_cam0.jpg


In [8]:
import json
import os
import glob
import pandas as pd
import numpy as np

# --- CONFIGURATION ---
UNITY_DIR = '/content/dataset_unity'  # Where your images actually are
OUTPUT_CSV = '/content/unity_fixed.csv'

# Virtual Screen Settings (Same as before)
VIRTUAL_DISTANCE_MM = 600
SCREEN_WIDTH_MM = 530
SCREEN_HEIGHT_MM = 300

def parse_vector_string(vec_str):
    # Converts string "(0.41, -0.31, ...)" to numpy array
    clean = vec_str.replace('(', '').replace(')', '')
    return np.fromstring(clean, sep=',')

data_list = []

# 1. Find all JSONs in the Colab folder
# We look recursively just in case they are in subfolders
search_pattern = os.path.join(UNITY_DIR, '**', '*.json')
json_files = glob.glob(search_pattern, recursive=True)

print(f"Found {len(json_files)} JSON files. Regenerating CSV...")

if len(json_files) == 0:
    print("ERROR: No JSON files found in /content/dataset_unity.")
else:
    for filepath in json_files:
        try:
            with open(filepath, 'r') as f:
                data = json.load(f)

            # 2. Extract Vector & Calculate Coords
            look_vec = parse_vector_string(data['eye_details']['look_vec'])
            vx, vy, vz = look_vec[0], look_vec[1], look_vec[2]

            screen_x_mm = vx * (VIRTUAL_DISTANCE_MM / abs(vz))
            screen_y_mm = vy * (VIRTUAL_DISTANCE_MM / abs(vz))

            norm_x = np.clip((screen_x_mm / SCREEN_WIDTH_MM) + 0.5, 0.0, 1.0)
            norm_y = np.clip(0.5 - (screen_y_mm / SCREEN_HEIGHT_MM), 0.0, 1.0)

            # 3. THE FIX: Construct the correct filename
            # "1.json"  -->  "1"  -->  "1_cam0.jpg"
            # We need to handle the full path to check existence, but save only the filename for the CSV
            
            # Get the directory of the json file
            json_dir = os.path.dirname(filepath)
            # Get the filename without extension (e.g., "1")
            base_name = os.path.splitext(os.path.basename(filepath))[0]
            
            # Construct the image filename
            img_filename = f"{base_name}_cam0.jpg"
            full_img_path = os.path.join(json_dir, img_filename)

            # 4. Verify Image Exists
            if os.path.exists(full_img_path):
                data_list.append([img_filename, norm_x, norm_y])
            else:
                # Fallback: Check for png just in case
                img_filename_png = f"{base_name}_cam0.png"
                full_img_path_png = os.path.join(json_dir, img_filename_png)
                if os.path.exists(full_img_path_png):
                    data_list.append([img_filename_png, norm_x, norm_y])

        except Exception as e:
            pass # Skip broken files

    # 6. Save new CSV
    df = pd.DataFrame(data_list, columns=['filename', 'x_norm', 'y_norm'])
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Success! Created {OUTPUT_CSV} with {len(df)} valid samples.")

Found 5000 JSON files. Regenerating CSV...
✅ Success! Created /content/unity_fixed.csv with 5000 valid samples.


In [14]:
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset, Subset
from torchvision import transforms
import pandas as pd
from PIL import Image
import numpy as np
import os

# --- 1. DEFINE THE DATASET CLASS ---
class EyeDataset(Dataset):
    def __init__(self, csv_file, root_dir, is_real_data=False):
        self.data_frame = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.is_real_data = is_real_data
        
        # Transform for UNITY (Zoom & Crop)
        self.transform_unity = transforms.Compose([
            transforms.CenterCrop(300),
            transforms.Resize((64, 64)),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        # Transform for REAL (Resize only)
        self.transform_real = transforms.Compose([
            transforms.Resize((64, 64)),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        row = self.data_frame.iloc[idx]
        filename = row.iloc[0]

        # --- PATH FIXING LOGIC ---
        if self.is_real_data:
            # FIX: Real data needs path reconstruction
            parts = filename.replace('\\', '/').split('/')
            clean_path = os.path.join(parts[-4], parts[-3], parts[-2], parts[-1])
            img_name = os.path.join(self.root_dir, clean_path)
            transform = self.transform_real
        else:
            # UNITY FIX: Ensure we use the root dir directly
            img_name = os.path.join(self.root_dir, filename)
            transform = self.transform_unity

        try:
            image = Image.open(img_name)
            image = transform(image)
        except Exception as e:
            # PRINT THE ERROR so we see exactly what path failed
            if idx < 5: # Only print first few errors to avoid spam
                print(f"FAIL: Could not load {img_name}")
            image = torch.zeros((3, 64, 64))

        coords = row.iloc[1:3].values.astype('float32')
        return image, torch.tensor(coords)

# --- 2. LOAD DATASETS (WITH FORCED PATHS) ---
unity_csv_fixed = '/content/unity_fixed.csv'
unity_root_fixed = '/content/dataset_unity'

print(f"Loading Unity from: {unity_root_fixed}")
print(f"Using CSV: {unity_csv_fixed}")

if not os.path.exists(unity_csv_fixed):
    print("CRITICAL: unity_fixed.csv is missing!")
else:
    dataset_unity = EyeDataset(csv_file=unity_csv_fixed, root_dir=unity_root_fixed, is_real_data=False)
    print(f"Unity Dataset Loaded: {len(dataset_unity)} samples")

# Real Dataset
dataset_real = None
real_csv = '/content/drive/MyDrive/FYP_ML/labels_real_full.csv'
real_root = '/content/dataset_real'

if os.path.exists(real_csv):
    dataset_real = EyeDataset(csv_file=real_csv, root_dir=real_root, is_real_data=True)
    full_dataset = ConcatDataset([dataset_unity, dataset_real])
    print(f"Real Dataset Loaded. Combined Size: {len(full_dataset)}")
else:
    full_dataset = dataset_unity
    print("Real CSV not found. Using Unity Only.")

# Limit Data
TOTAL_LIMIT = 40000 
if len(full_dataset) > TOTAL_LIMIT:
    indices = torch.randperm(len(full_dataset))[:TOTAL_LIMIT]
    train_dataset = Subset(full_dataset, indices)
else:
    train_dataset = full_dataset

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)

# --- 3. IMMEDIATE VERIFICATION TEST ---
try:
    images, labels = next(iter(train_loader))
    print(f"SUCCESS! Loaded a batch of shape: {images.shape}")
except Exception as e:
    print(f"STILL FAILING: {e}")

Loading Unity from: /content/dataset_unity
Using CSV: /content/unity_fixed.csv
✅ Unity Dataset Loaded: 5000 samples
✅ Real Dataset Loaded. Combined Size: 2648936

🧪 TESTING: Attempting to load first batch...
🎉 SUCCESS! Loaded a batch of shape: torch.Size([128, 3, 64, 64])
You may now run the Training Loop cell.


In [15]:
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
import time

# --- 1. SETUP MODEL ---
model = models.resnet18(pretrained=True)

# Modify the last layer (The output head)
# Original ResNet outputs 1000 classes (for ImageNet)
# We change it to output 2 numbers (X, Y coordinates)
model.fc = nn.Linear(model.fc.in_features, 2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Model loaded on: {device}")

# 2. Define Loss and Optimizer
criterion = nn.MSELoss() # Mean Squared Error (Standard for coordinates)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3. Training Function
def train_model(num_epochs=5):
    model.train()
    
    for epoch in range(num_epochs):
        running_loss = 0.0
        start_time = time.time()
        
        for i, (images, labels) in enumerate(train_loader):
            images = images.to(device)
            labels = labels.to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            if (i+1) % 50 == 0:
                print(f"   Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")
                
        epoch_loss = running_loss / len(train_loader)
        duration = time.time() - start_time
        print(f"--- Epoch {epoch+1} Finished. Avg Loss: {epoch_loss:.5f} (Time: {duration:.1f}s) ---")

train_model(num_epochs=5)

Model loaded on: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


   Step [50/313], Loss: 0.0207
   Step [100/313], Loss: 0.0196
   Step [150/313], Loss: 0.0205
   Step [200/313], Loss: 0.0167
   Step [250/313], Loss: 0.0138
   Step [300/313], Loss: 0.0178
--- Epoch 1 Finished. Avg Loss: 0.10070 (Time: 47.5s) ---
   Step [50/313], Loss: 0.0153
   Step [100/313], Loss: 0.0157
   Step [150/313], Loss: 0.0168
   Step [200/313], Loss: 0.0152
   Step [250/313], Loss: 0.0171
   Step [300/313], Loss: 0.0129
--- Epoch 2 Finished. Avg Loss: 0.01616 (Time: 31.9s) ---
   Step [50/313], Loss: 0.0179
   Step [100/313], Loss: 0.0153
   Step [150/313], Loss: 0.0148
   Step [200/313], Loss: 0.0168
   Step [250/313], Loss: 0.0175
   Step [300/313], Loss: 0.0154
--- Epoch 3 Finished. Avg Loss: 0.01550 (Time: 33.0s) ---
   Step [50/313], Loss: 0.0143
   Step [100/313], Loss: 0.0155
   Step [150/313], Loss: 0.0177
   Step [200/313], Loss: 0.0168
   Step [250/313], Loss: 0.0143
   Step [300/313], Loss: 0.0161
--- Epoch 4 Finished. Avg Loss: 0.01526 (Time: 31.8s) ---
   S

In [19]:
# 1. Install the widget library (Required for the workaround)
!pip install anywidget

import anywidget
import traitlets
import os
import base64
import mimetypes
from google.colab import drive

# --- DEFINING THE DOWNLOADER WIDGET ---
class FileDownloader(anywidget.AnyWidget):
    file_path = traitlets.Unicode(help="Path to the file to be downloaded").tag(sync=True)
    button_text = traitlets.Unicode("Download File").tag(sync=True)

    _esm = """
    export function render({ model, el }) {
      let btn = document.createElement("button");
      btn.classList.add("jupyter-widgets", "jupyter-button", "widget-button");
      btn.style.width = "100%";
      btn.innerText = "Waiting for Kernel...";
      btn.disabled = true;

      model.on("change:button_text", () => {
        if (!btn.disabled) { btn.innerText = model.get("button_text"); }
      });

      btn.addEventListener("click", () => {
        const filePath = model.get("file_path");
        if (!filePath) { alert("No file path set!"); return; }
        
        const originalText = btn.innerText;
        btn.innerText = "Downloading...";
        btn.disabled = true;
        model.send({ type: "request_download" });

        const restoreBtn = () => { btn.innerText = originalText; btn.disabled = false; };
        setTimeout(restoreBtn, 5000);
      });

      el.appendChild(btn);

      model.on("msg:custom", (msg) => {
        if (msg.type === "connection_verified") {
            btn.disabled = false;
            btn.innerText = model.get("button_text");
        }
        else if (msg.type === "file_content") {
            const byteCharacters = atob(msg.content);
            const byteNumbers = new Array(byteCharacters.length);
            for (let i = 0; i < byteCharacters.length; i++) {
                byteNumbers[i] = byteCharacters.charCodeAt(i);
            }
            const byteArray = new Uint8Array(byteNumbers);
            const blob = new Blob([byteArray], { type: msg.mime_type });
            const url = window.URL.createObjectURL(blob);
            const a = document.createElement("a");
            a.style.display = "none";
            a.href = url;
            a.download = msg.filename;
            document.body.appendChild(a);
            a.click();
            window.URL.revokeObjectURL(url);
            document.body.removeChild(a);
            
            btn.innerText = model.get("button_text");
            btn.disabled = false;
        } else if (msg.type === "error") {
            alert(`Error: ${msg.message}`);
            btn.innerText = model.get("button_text");
            btn.disabled = false;
        }
      });

      setTimeout(() => { model.send({ type: "check_connection" }); }, 500);
    }
    """

    def __init__(self, file_path=None, **kwargs):
        super().__init__(**kwargs)
        if file_path:
            self.file_path = file_path
        self.on_msg(self._handle_custom_msg)

    def _handle_custom_msg(self, msg, content):
        msg_type = msg.get("type")
        if msg_type == "check_connection":
            self.send({"type": "connection_verified"})
        elif msg_type == "request_download":
            self._process_download()

    def _process_download(self):
        target_path = self.file_path
        if not target_path or not os.path.exists(target_path):
            self.send({"type": "error", "message": f"File not found: {target_path}"})
            return
        try:
            mime_type, _ = mimetypes.guess_type(target_path)
            if mime_type is None: mime_type = 'application/octet-stream'
            with open(target_path, "rb") as f:
                file_content = f.read()
            b64_content = base64.b64encode(file_content).decode("utf-8")
            self.send({
                "type": "file_content",
                "filename": os.path.basename(target_path),
                "mime_type": mime_type,
                "content": b64_content
            })
        except Exception as e:
            self.send({"type": "error", "message": str(e)})

# --- GENERATE BUTTONS ---
print("👇 Click these buttons to download your models 👇")

# Button 1: The Web Model (ONNX)
display(FileDownloader(file_path="gaze_model.onnx", button_text="Download gaze_model.onnx (Web Ready)"))

# Button 2: The Backup Model (PyTorch)
display(FileDownloader(file_path="gaze_model.pth", button_text="Download gaze_model.pth (Backup)"))

👇 Click these buttons to download your models 👇


In [24]:
import torch
import torch.onnx
import os

# 1. Define the Dummy Input (Fixed size 64x64)
dummy_input = torch.randn(1, 3, 64, 64).to(device)

print("🔄 Attempting Stable Export (Opset 11, Static Shapes)...")

# 2. Export with "Safe Mode" settings
# - opset_version=11 (Most compatible, rarely crashes)
# - NO dynamic_axes (Fixed 64x64 input is safer for Web MVPs)
torch.onnx.export(
    model, 
    dummy_input, 
    "gaze_model.onnx", 
    export_params=True,
    opset_version=11, 
    do_constant_folding=True,
    input_names=['input_image'], 
    output_names=['gaze_coords']
)

# 3. Verify File Size
file_size = os.path.getsize("gaze_model.onnx") / (1024 * 1024) # Convert to MB
print(f"📊 New File Size: {file_size:.2f} MB")

if file_size < 10:
    print("❌ ERROR: File is still too small! The export failed silently.")
else:
    print("✅ SUCCESS: File looks correct (approx 40-45 MB).")
    
    # 4. Download
    print("👇 Click below to download 👇")
    display(FileDownloader(file_path="gaze_model.onnx", button_text="Download Corrected Model"))

W0205 01:18:32.859000 1605 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


🔄 Attempting Stable Export (Opset 11, Static Shapes)...


W0205 01:18:33.357000 1605 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0205 01:18:33.359000 1605 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0205 01:18:33.361000 1605 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0205 01:18:33.363000 1605 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 127, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 122, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/adapters/axes_input_to_attribute.h:65: adapt: Asserti

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 40 of general pattern rewrite rules.
📊 New File Size: 0.09 MB
❌ ERROR: File is still too small! The export failed silently.


In [ ]:
# 1. Install Gradio
!pip install gradio

import gradio as gr
import torch
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
from PIL import Image

# --- 1. LOAD THE BRAIN (PyTorch) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Re-create the empty model architecture
model = models.resnet18(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2) # X, Y output

# Load the weights you trained
# Map location ensures it works even if you switched from GPU to CPU
model.load_state_dict(torch.load("gaze_model.pth", map_location=device))
model.to(device)
model.eval() # Set to "Test Mode" (turns off dropout, etc.)

print("✅ Model Loaded Successfully!")

# --- 2. DEFINE THE PROCESSING PIPELINE ---
# This must match exactly what we did during training (Resize -> Normalize)
transform_pipeline = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict_gaze(image):
    if image is None:
        return "No face detected"
    
    # Convert Gradio image (numpy) to PIL
    pil_image = Image.fromarray(image)
    
    # Transform
    input_tensor = transform_pipeline(pil_image).unsqueeze(0) # Add batch dimension
    input_tensor = input_tensor.to(device)
    
    # Predict
    with torch.no_grad():
        output = model(input_tensor)
        
    # Extract numbers
    x_pred = output[0, 0].item()
    y_pred = output[0, 1].item()
    
    return f"X: {x_pred:.2f}, Y: {y_pred:.2f}"

# --- 3. LAUNCH THE UI ---
interface = gr.Interface(
    fn=predict_gaze,
    inputs=gr.Image(sources=["webcam"], streaming=True), # Enable streaming for "Real-time" feel
    outputs="text",
    live=True,
    title="Eye Gaze Tracker (PyTorch Backend)",
    description="Look at the camera. The model predicts where you are looking (0.0 - 1.0)."
)

interface.launch(debug=True, share=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


✅ Model Loaded Successfully!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://358eff6209208a7e4d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
